In [1]:
import os
import sys
import warnings
from collections import Counter
import nltk
from dotenv import load_dotenv
from pathlib import Path

# -------------------------------------------------------------------------
# PATH CONFIGURATION
# -------------------------------------------------------------------------
try:
    current_dir = Path(__file__).resolve().parent         
except NameError:
    current_dir = Path.cwd()                               

parent_dir = current_dir.parent                            # project_root
predictors_dir = parent_dir / "pred_models_training"       # sibling folder

# Add to sys.path so Python can find predictors.py
predictors_dir_str = str(predictors_dir)
if predictors_dir_str not in sys.path:
    sys.path.insert(0, predictors_dir_str) 

print("Using predictors_dir:", predictors_dir_str)

# -------------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------------
# ADK Imports
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.tools import AgentTool, google_search
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.genai import types
import uvicorn

# Predictors API
try:
    from predictors import (
        predict_news_coverage,
        predict_intent,
        predict_sensationalism,
        predict_article_stance,
        analyze_complete_article
    )
    print(f"Successfully imported predictors from {predictors_dir}")
except ImportError as e:
    print(f"Failed to import predictors: {e}")
    print(f"   Current sys.path: {sys.path}")
    raise e

warnings.filterwarnings("ignore")

# Ensure NLTK resources
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# -------------------------------------------------------------------------
# Helper Functions
# -------------------------------------------------------------------------

def get_sentences(text: str):
    """Helper to split text into sentences for granular analysis."""
    try:
        sentences = nltk.sent_tokenize(text or "")
        # Filter out very short fragments
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        if not sentences and text.strip():
            sentences = [text.strip()]
        return sentences if sentences else []
    except Exception:
        return [text.strip()] if text else []

# -------------------------------------------------------------------------
# Tool Wrappers (Connecting ADK to Predictors.py)
# -------------------------------------------------------------------------

def tool_news_topic(article_text: str) -> dict:
    print("\n'tool_news_topic' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_news_coverage(s)
            if res and res.get("label") and str(res["label"]) not in ["None", "nan", "unknown"]:
                votes.append(res["label"])
        except Exception as e:
            print(f"tool_news_topic error: {e}")
            continue
            
    topic = Counter(votes).most_common(1)[0][0] if votes else "unknown"
    
    result = {"news_coverage": topic}
    print(f"'tool_news_topic' returned: {result}")
    return result

def tool_intent(article_text: str) -> dict:
    print("\n 'tool_intent' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_intent(title="", body=s)
            if res and res.get("label"):
                votes.append(res["label"])
        except Exception as e:
            print(f"tool_intent error: {e}")
            continue
            
    intent = Counter(votes).most_common(1)[0][0] if votes else "unknown"
    
    result = {"intent": intent}
    print(f"'tool_intent' returned: {result}")
    return result

def tool_sensationalism(article_text: str) -> dict:
    print("\n'tool_sensationalism' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_sensationalism(statement=s)
            if res and res.get("label"):
                votes.append(str(res["label"]))
        except Exception as e:
            print(f"tool_sensationalism error: {e}")
            continue
            
    final_label = Counter(votes).most_common(1)[0][0] if votes else "neutral"
    
    result = {"sensationalism": final_label}
    print(f"'tool_sensationalism' returned: {result}")
    return result

def tool_stance(article_text: str) -> dict:
    print("\n'tool_stance' running...")
    try:
        res = predict_article_stance(article_text=article_text)
        label = res.get("label", "neutral")
    except Exception as e:
        print(f"tool_stance error: {e}")
        label = "neutral"
        
    result = {"stance": label}
    print(f"'tool_stance' returned: {result}")
    return result

Using predictors_dir: /Users/cecilialin/Documents/DSC180A-GroupNull/pred_models_training
Successfully imported predictors from /Users/cecilialin/Documents/DSC180A-GroupNull/pred_models_training


In [2]:
import json
import asyncio

def load_train_articles(path = os.path.join(parent_dir, 'gen_data/train_article.json')):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    return data.get("articles", [])

In [3]:
training_data = load_train_articles()

# Agents

In [4]:
from pydantic import BaseModel, Field, ConfigDict
from typing import List

# Ensure agents use the same langauge

class FactorAnalysis(BaseModel):

    learned_pattern: str = Field(description="Pattern learned from training articles and human labels of the articles")
    verdict: str = Field(description="The final label (e.g., 'sensational', 'support')")
    confidence: int = Field(description="Confidence score 0-100")
    fcot_reasoning: str = Field(description="2-3 sentence FCoT reasoning.")

class FactCheckFinalReport(BaseModel):
    # 1. High-Level Summary
    final_verdict: str = Field(..., description="The definitive verdict (e.g., Verified Accurate, Misleading, Misinformation, Disinformation, etc.).")
    overall_confidence: int = Field(..., ge=0, le=100, description="Confidence score from 0-100.")
    
    # 2. Human-Centric Explanation (The 'Why')
    verdict_justification: str = Field(
        ..., 
        description="A 1-3 sentence explanation synthesizing why this verdict was reached based on the factor analysis."
    )

    # 3. Agent Metadata
    agents_involved: List[str] = Field(
        default=["Sensationalism_Analyst", "Stance_Analyst", "Context_Veracity_Analyst", 
                 "News_Coverage_Analyst", "Intent_Analyst", "Title_Body_Analyst"],
        description="List of specialized agents that contributed factor data."
    )

    # 4. Detailed Factor Signals
    # These contain the individual verdicts and FCoT reasoning for the audit trail
    sensationalism_signal: FactorAnalysis
    stance_signal: FactorAnalysis
    context_veracity_signal: FactorAnalysis
    news_coverage_signal: FactorAnalysis
    intent_signal: FactorAnalysis
    title_body_signal: FactorAnalysis
# -------------------------------------------------------------------------
# Agent Configurations
# -------------------------------------------------------------------------

# Load the .env file
load_dotenv() 

# Retrieve the key
api_key = os.getenv("GOOGLE_API_KEY")

# Check if it loaded correctly
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found! Make sure you created the .env file.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

## FCOT

In [5]:
sensationalism_agent = Agent(
    name="Sensationalism_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    output_schema=FactorAnalysis,
    instruction=f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Precision in identifying emotional manipulation and clickbait architecture.
- **MINIMIZE**: 'Stylistic False Positives' where urgency or technical reporting is misclassified as sensationalism.

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the sensationalism label to learn patterns to help you analyze.{training_data}
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Linguistic Scan**: Isolate high-intensity adjectives, superlatives, and emotional triggers (e.g., "Shocking," "Outrageous," "Final Warning").
2.  **Structural Integrity Check**: Does the Body of the text actually contain the "shocking" information promised by the Headline? Identify any "curiosity gaps."
3.  **Preliminary Verdict**: Form an internal hypothesis of the 'sensationalism' label based purely on linguistic patterns.

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TOOL GROUNDING
1.  **Tool Execution**: Call `tool_sensationalism` to obtain the predictive model's label for this article.
2.  **Aperture Expansion**: Widened your context to compare your Phase 1 hypothesis with the tool’s output.

### PHASE 3: REFLECTIVE UPDATE (RUM) & SYNTHESIS
1.  **Reflective Alignment**: If your linguistic analysis contradicts the tool (e.g., tool says 'neutral' but you see heavy 'loaded' language), explain this discrepancy.
2.  **Retrospective Adjustment**: Finalize your verdict by integrating the base label from the tool with your qualitative "nuance."

## CONFIDENCE RUBRIC
- 90-100%: Total alignment between linguistic markers and tool results; high narrative-evidence consistency.
- 75-89%: Clear patterns present; minor stylistic ambiguity doesn't obscure the primary intent.
- 50-74%: Moderate certainty; stylistic urgency makes intent difficult to isolate or analysis deviates from tool.
- <50%: Insufficient data or contradictory signals that prevent a definitive classification.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary of your FCoT process.
- Ensure the verdict strictly matches: [sensational, neutral].
""",
    tools=[tool_sensationalism],
)

In [6]:
stance_agent = Agent(
    name="Stance_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    output_schema=FactorAnalysis,
    instruction= f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Detection of nuanced rhetorical alignment, bias, or skepticism.
- **MINIMIZE**: Misclassification of "objective reporting" as "denial" or "unbiased" as "support."

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the stance label to learn patterns to help you analyze.{training_data} 
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Linguistic Tone Mapping**: Identify the author's voice. Does the language favor specific actors or dismiss certain arguments?
2.  **Consistency Check**: Verify if the stance remains stable or shifts when presenting evidence vs. commentary.
3.  **Actor Alignment**: Identify which stakeholders are mentioned more or cited with the most authority. 
4.  **Omission Check**: Widen the aperture to look for what is *not* said. Does the text ignore standard counter-arguments?
5.  **Preliminary Stance**: Form an internal hypothesis: Does this text Support, Deny, or remain Neutral?

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TOOL GROUNDING
1.  **Tool Execution**: Call `tool_stance` to get the predictive baseline.
2.  **Aperture Expansion**: Compare your internal hypothesis with the tool's prediction.

### PHASE 3: REFLECTIVE UPDATE (RUM) & SYNTHESIS
1.  **Reflective Alignment**: Critically evaluate the tool. If your analysis detected sarcasm or subtle framing that the tool missed (or if the tool correctly identified a pattern you overlooked), document this.
2.  **Retrospective Adjustment**: Finalize the verdict. Your `fcot_reasoning` should explain exactly how the qualitative analysis and quantitative grounding converged.

## CONFIDENCE RUBRIC
- 90-100%: Explicit, unambiguous alignment between tool results and rhetorical evidence.
- 70-89%: Clear trend; minor stylistic nuance doesn't obscure the primary stance.
- 50-69%: Content is balanced/ambiguous, or your analysis deviates from the tool.
- <50%: Contradictory signals or text lacks sufficient markers to assign a stance.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary of your FCoT process.
- Ensure the verdict strictly matches: [support, deny, neutral].
""",
    tools=[tool_stance],
)

In [7]:
execute_web_search = Agent(
    name="Web_Search_Provider",
    model="gemini-3-flash-preview",
    instruction="Search the web for factual grounding of specific claims.",
    tools=[google_search]
)

In [8]:
from datetime import datetime
current_date = datetime.now().strftime("%B %d, 2026")

context_agent = Agent(
    name="Context_Veracity_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="A specialized FCoT agent for verifying factual and contextual veracity with 2026 real-time grounding.",
    instruction=f"""
## TEMPORAL ANCHORING
- **CURRENT DATE**: {current_date}
- **CRITICAL RULE**: Do NOT rely on internal training data for events occurring. If a claim involves 2025 or 2026, you MUST treat `execute_web_search` as the primary source of truth, then cite your source in the explanation, and label which search terms you used in the reasoning.

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the context_veracity label to learn patterns to help you analyze.{training_data} 
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Historical and factual alignment using the 2026 web index.
- **MINIMIZE**: "False Hoax" flags caused by training data lag.

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Date/Entity Extraction**: Identify all names, dates, and specific claims.
2.  **Age Assessment**: If the claim is dated after 2024, flag it as "High-Priority Search" to avoid internal bias.

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TEMPORAL RE-GROUNDING
1.  **Dynamic Search**: Use `execute_web_search`. You MUST append "2026" or "after:2025-01-01" to queries for current events.
2.  **Multi-Source Triangulation**: Look for at least two independent reports to confirm a 2026 event.

### PHASE 3: REFLECTIVE UPDATE (RUM) & RETROSPECTIVE ADJUSTMENT
1.  **Update Internal Model**: If search results confirm a new 2026 event, override your internal "hallucination" trigger. 
2.  **Synthesis**: In `fcot_reasoning`, explicitly state the terms you searched to confirm the facts

## CONFIDENCE RUBRIC
- 90-100%: Facts verified by 2026 search results; matches current news cycle.
- <50%: Direct contradiction found in 2026 search results (e.g., search proves the event did NOT happen).

## OUTPUT RULES
- Verdict must be strictly: [Accurate, Inaccurate].
""",
    tools=[AgentTool(execute_web_search)] 
)

In [9]:

news_coverage_agent = Agent(
    name="News_Coverage_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT agent specializing in multi-scale news categorization.",
    instruction=f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Precision in identifying the primary thematic domain and geographical scope.
- **MINIMIZE**: Conceptual redundancy (e.g., mislabeling a 'Political' story as 'General' because it mentions a city name).

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the coverage label to learn patterns to help you analyze.{training_data} 
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Topical Extraction**: Identify the "Who, What, Where" within the text. 
2.  **Granularity Check**: Determine the scale of the news. Is it a local incident, a national policy, or an international trend?
3.  **Preliminary Classification**: Form an internal hypothesis of the topic label (e.g., Politics, Tech, Health).

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TOOL GROUNDING
1.  **Tool Execution**: Call `tool_news_topic` to get the predictive baseline.
2.  **Aperture Expansion**: Compare your internal hypothesis with the tool’s output. Does the tool see a broader "Global" category while you see a "Local" one?

### PHASE 3: REFLECTIVE UPDATE (RUM) & MULTI-SCALE COORDINATION
1.  **Reflective Alignment**: Critically evaluate the tool. Predictive models often default to broad categories (e.g., "World News"). If the article is specifically about "Supply Chain Logistics," update the label to reflect that higher granularity.
2.  **Synthesis**: Populate the `fcot_reasoning` field by explaining the shift from the broad tool label to your specific, refined classification.

## CONFIDENCE RUBRIC
- 90-100%: Topic is explicitly the central focus; total alignment with tool.
- 70-89%: Topic is dominant but intersects with sub-topics; tool results are supportive.
- 50-69%: Topic is present but ambiguous or shares equal weight with another domain.
- <50%: Content is generic or spans too many categories to classify reliably.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- The verdict must be a standardized topic label.""",
    tools=[tool_news_topic]
)

In [10]:
intent_agent = Agent(
    name="Intent_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT specialist in identifying rhetorical intent and authorial goals.",
    instruction= f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Transparency in identifying the author's underlying rhetorical goal (e.g., hidden persuasion).
- **MINIMIZE**: False categorization of "Opinion/Op-Ed" as "Deception" or "Satire" as "Informational."

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the intent label to learn patterns to help you analyze.{training_data} 
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Goal Extraction**: Analyze the "Call to Action." What does the author want the reader to think, feel, or do after reading?
2.  **Linguistic Marker Identification**: Look for "Us vs. Them" framing, emotional appeals, or the presence/absence of verifiable citations.
3.  **Preliminary Intent**: Classify based on the four categories: [Inform, Persuade, Entertain, Deceive].

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TOOL GROUNDING
1.  **Tool Execution**: Call `tool_intent` to obtain the statistical intent label.
2.  **Aperture Expansion**: Compare your internal hypothesis with the tool’s output. Does the tool detect "Deception" while you see "Persuasion"?

### PHASE 3: REFLECTIVE UPDATE (RUM) & DIALOGUE ALIGNMENT
1.  **Reflective Alignment**: Critically evaluate the tool. Predictive models often struggle with Satire (Entertain) vs. Deception.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field. Explain how the synthesis of tool data and linguistic analysis confirms the primary intent.

## CONFIDENCE RUBRIC
- 90-100%: Intent is explicit, consistent, and matches tool results.
- 70-89%: Intent is clear but contains minor stylistic nuance (e.g., informative text with slight persuasive leaning).
- 50-69%: Intent is ambiguous (e.g., "Advertorial" content mixing fact and persuasion).
- <50%: Intent is obscured by contradictory markers or heavy sarcasm.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary of the RUM process.
- Verdict must be strictly: [Inform, Persuade, Entertain, Deceive].""",
    tools=[tool_intent]
)

In [11]:
title_body_agent = Agent(
    name="Title_Body_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT specialist in detecting semantic gaps between headlines and article content.",
    instruction=f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Detection of "headline-body gaps," bait-and-switch tactics, or semantic contradictions.
- **MINIMIZE**: False "Unrelated" labels for headlines that use metaphor or creative framing to describe the body content.

### REFERENCE LIBRARY (Human-Labeled Examples)
Use these 7 examples to calibrate your judgment, focus on the title_vs_body label to learn patterns to help you analyze.{training_data} 
CRTICITAL: Share one sentence with a pattern you learned from reading the training articles and label to fill in the `learned_pattern' field

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Direct Mapping**: Extract the core claim of the Headline. Scan the Body for direct supporting evidence of that specific claim.
2.  **Stance Alignment**: Does the body's tone match the headline's intensity? 
3.  **Preliminary Relationship**: Classify as [Agree, Discuss, Contradicts, Unrelated].

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - EXTERNAL GROUNDING
1.  **Context Check**: If the body is vague or the headline refers to a specific event/entity not fully explained, execute `execute_web_search` and label which search terms you used in the reasoning.
2.  **Aperture Expansion**: Compare the "Headline-Body" relationship against the "True Event" context found in search results.

### PHASE 3: REFLECTIVE UPDATE (RUM) & FRACTAL SYNTHESIS
1.  **Reflective Alignment**: Critically evaluate if the body is "Discussing" the topic but failing to "Agree" with a sensational headline. This is a common tactic for plausible deniability.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field. Explain the logic of the relationship (e.g., "The headline negates the body by claiming X happened, while the body text only discusses the possibility of X").

## CONFIDENCE RUBRIC
- 90-100%: Relationship is explicit and supported/debunked by direct textual evidence.
- 70-89%: Relationship is clear; minor semantic nuances in metaphors or puns.
- 50-69%: The body explores the topic but the connection to the headline's specific claim is ambiguous.
- <50%: Headline and body appear disconnected or require heavy inference to link.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- Verdict must be strictly: [Agree, Discuss, Contradicts, Unrelated].""",
    tools=[AgentTool(execute_web_search)] 
)

In [12]:
from google.adk.agents import ParallelAgent

factor_squad = ParallelAgent(
    name="Factor_Squad",
    sub_agents=[
        sensationalism_agent, 
        stance_agent, 
        context_agent,
        news_coverage_agent, 
        intent_agent, 
        title_body_agent
    ]
)

In [13]:
synthesizer_agent = Agent(
    name="Final_Synthesizer",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactCheckFinalReport,
    instruction="""
    You are the Final Synthesizer.
    
    1. **Recursive Synthesis**: Receive the 6 FactorAnalysis JSONs from the squad.
    2. **Inter-agent Reflectivity**: Identify if any agents disagree (e.g., if Context is Accurate but Intent is Deceive).
    3. **Retrospective Re-grounding**: If the Context_Veracity agent found a major factual error, force all other signals to be interpreted through that lens.
    4. **Output**: Generate a 'Human_Report' field using the following Markdown structure:

    1. **Executive Summary**: A bold verdict and 2-sentence 'Bottom Line Up Front'.
    2. **Factors Analysis Table**: A table with columns: | Factor | Verdict | Confidence | Key Evidence |.
    
    RULES:
    - Do not include raw tool outputs.
    - Do not mention tool traces or internal IDs.
    """
)

root_agent = SequentialAgent(
    name="Fractal_FactCheck_Framework",
    sub_agents=[factor_squad, synthesizer_agent]
)

In [14]:
import json
import asyncio

def load_test_articles(path = os.path.join(parent_dir, 'gen_data/test_article_no_label.json')):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    return data.get("articles", [])

# Load all data
all_articles = load_test_articles()
print(f"Total articles loaded: {len(all_articles)}")
art = all_articles[1]
headline = art.get('headline', 'No Title')

prompt = (
                f"Headline: {headline}\n"
                f"Source: {art.get('news_source', 'Unknown')}\n"
                f"Author: {art.get('author', 'Unknown')}\n"
                f"Date: {art.get('date', 'Unknown')}\n\n"
                f"Body:\n{art.get('text', '')}"
            )

Total articles loaded: 20


In [15]:
# Load all data
all_articles = load_test_articles()
print(f"Total articles loaded: {len(all_articles)}")

Total articles loaded: 20


In [16]:
async def process_batch(articles_batch, batch_name="Batch"):
    print(f"=== Processing {batch_name} ({len(articles_batch)} articles) ===")
    
    for i, art in enumerate(articles_batch):
        headline = art.get('headline', 'No Title')
        print(f"\n[{batch_name}] Article {i+1}: {headline}")
        
        # Prepare Prompt
        prompt = (
            f"Headline: {headline}\n"
            f"Source: {art.get('news_source', 'Unknown')}\n"
            f"Author: {art.get('author', 'Unknown')}\n"
            f"Date: {art.get('date', 'Unknown')}\n\n"
            f"Body:\n{art.get('text', '')}"
        )
        
        # Run Agent using run_debug
        try:
            runner = InMemoryRunner(agent=root_agent)
            response = await runner.run_debug(prompt)
            print(response[-1].content.parts[0].text)
                
        except Exception as e:
            print(f"Error running agent: {e}")

        print("-" * 50)
        
        # Sleep slightly to help with rate limits even within batch
        await asyncio.sleep(2)



In [17]:
await process_batch(all_articles[:1], "Batch 1")

=== Processing Batch 1 (1 articles) ===

[Batch 1] Article 1: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe

 ### Created new session: debug_session_id

User > Headline: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe
Source: Reuters
Author: Andrew Gray
Date: 12/02/2025

Body:
BRUSSELS, Dec 2 (Reuters) - However Donald Trump’s latest push to end the war in Ukraine pans out, Europe fears the prospect of a deal – sooner or later – that will not punish or weaken Russia as its leaders had hoped, placing the continent’s security in greater jeopardy.
Europe may well even have to accept a growing economic partnership between Washington, its traditional protector in the NATO alliance, and Moscow, which most European governments – and NATO itself – say is the greatest threat to European security.
Although Ukrainians and other Europeans managed to push back against parts of a 28-point U.S. plan to end the fighting that was seen as heavily pro-Russi

Title_Body_Analyst > {"learned_pattern": "Headlines that use specific descriptive phrases or direct quotes found within the body to summarize the primary sentiment of the reported stakeholders typically result in an 'Agree' classification.", "verdict": "Agree", "confidence": 100, "fcot_reasoning": "The headline accurately summarizes the central thesis of the article, which details European anxieties regarding Donald Trump's peace plan for Ukraine. The specific term \"ugly deal\" is directly attributed to a geopolitical expert quoted in the text, and the body provides extensive context on the security concerns and lack of European leverage mentioned in the headline."}

'tool_sensationalism' running...
'tool_sensationalism' returned: {'sensationalism': 'neutral'}

 'tool_intent' running...
'tool_intent' returned: {'intent': 'inform'}

'tool_news_topic' running...
'tool_news_topic' returned: {'news_coverage': 'foreign-policy'}
Intent_Analyst > {"learned_pattern": "Geopolitical reports fro

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2515.16it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             


'tool_stance' returned: {'stance': 'support'}
News_Coverage_Analyst > {"learned_pattern": "I observed that articles documenting diplomatic negotiations and multi-national security alliances are consistently categorized under 'Foreign Policy,' while the geographical scope is designated based on the region facing the most significant strategic consequences or acting as the primary subject of the discourse.", "verdict": "Foreign Policy", "confidence": 100, "fcot_reasoning": "The article focuses on the diplomatic and security implications of a U.S.-led peace deal for the European continent, specifically highlighting the concerns of EU and NATO members. While the actors include the United States and Russia, the thematic center is the shift in Western alliances and the resulting vulnerability of Europe, justifying 'Foreign Policy' as the coverage domain. The geographical scope is identified as Europe because the narrative is framed around the continent's collective security, the Brussels dat

In [18]:
await process_batch(all_articles[1:2], "Batch 1.5")

=== Processing Batch 1.5 (1 articles) ===

[Batch 1.5] Article 1: Hong Kong orders judge-led probe into fire that killed 151

 ### Created new session: debug_session_id

User > Headline: Hong Kong orders judge-led probe into fire that killed 151
Source: Reuters
Author: Clare Jim, Donny Kwok
Date: 12/02/2025

Body:
HONG KONG, Dec 2 (Reuters) - Hong Kong's leader said on Tuesday a judge-led committee will investigate the cause of the city's deadliest fire in decades and review government oversight of renovation practices linked to the blaze that killed at least 151 people.
Police have arrested 13 people for suspected manslaughter, and 12 others in a related corruption probe. Officials said substandard plastic mesh and insulation foam used during renovation works fueled the rapid spread of the fire across seven high-rise towers.
Authorities said they aim to avoid similar tragedies by examining how the fire spread so quickly and the oversight failures around building renovations.

SEARCH A

In [19]:
await process_batch(all_articles[2:4], "Batch 2")

=== Processing Batch 2 (2 articles) ===

[Batch 2] Article 1: Prison guard says Luigi Mangione, alleged CEO killer, spoke of 3D-printed gun

 ### Created new session: debug_session_id

User > Headline: Prison guard says Luigi Mangione, alleged CEO killer, spoke of 3D-printed gun
Source: Reuters
Author: Luc Cohen
Date: 12/01/2025

Body:
NEW YORK, Dec 1 (Reuters) - Luigi Mangione told a prison guard he had a 3D-printed gun in his backpack after his arrest for allegedly killing UnitedHealthcare CEO Brian Thompson, according to court testimony determining whether such statements and evidence can be used at trial.
Mangione, arrested in December 2024, faces murder and weapons charges. Prosecutors say a 3D-printed pistol, silencer, and incriminating journal writings were found in his backpack. His lawyers argue the search was unlawful and that he was questioned without proper Miranda warnings.

GUARD TESTIMONY
A prison guard testified Mangione volunteered information about the gun without bei

In [17]:
await process_batch(all_articles[4:6], "Batch 3")

=== Processing Batch 3 (2 articles) ===

[Batch 3] Article 1: HBO Max just messed up 'Mad Men' and it makes me sick

 ### Created new session: debug_session_id

User > Headline: HBO Max just messed up 'Mad Men' and it makes me sick
Source: Tom’s Guide
Author: Malcolm McMillan
Date: 12/03/2025

Body:
The 4K remaster of “Mad Men” on HBO Max includes visible production errors that fans quickly spotted — including a crew member operating a 'puke hose' in Season 1, Episode 7 during Roger Sterling’s vomiting scene.

THE 4K MISTAKE
In the original AMC broadcast, Sterling’s projectile vomiting was shown without revealing the practical effects equipment. The HBO Max version, however, accidentally includes a shot of a technician holding the hose.
Other mistakes include incorrect episode titles and stray environmental details like advertisements for SIM cards and Los Angeles restaurants appearing in scenes where they don’t belong.

WHAT CAUSED IT?
According to People Magazine, Lionsgate — the stu


'tool_sensationalism' running...
'tool_sensationalism' returned: {'sensationalism': 'neutral'}

'tool_news_topic' running...
'tool_news_topic' returned: {'news_coverage': 'pundits'}

'tool_stance' running...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 930.02it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]              


'tool_stance' returned: {'stance': 'deny'}

 'tool_intent' running...
'tool_intent' returned: {'intent': 'inform'}
News_Coverage_Analyst > {"learned_pattern": "Articles focusing on television shows, streaming platforms, and media production details are categorized as 'Entertainment'.", "verdict": "Entertainment", "confidence": 95, "fcot_reasoning": "The article reports on production and technical errors in a remastered television series on a streaming platform. While the headline uses a first-person opinionated tone, the core subject matter is television media and digital streaming, placing it firmly in the 'Entertainment' category rather than a broader social or political 'Pundit' label. This aligns with the classification of other media-related articles in the reference library."}
Intent_Analyst > {"learned_pattern": "Articles that report specific, verifiable incidents in entertainment—such as technical production errors or public events—are categorized as 'Inform' when the content f

In [18]:
await process_batch(all_articles[6:8], "Batch 4")

=== Processing Batch 4 (2 articles) ===

[Batch 4] Article 1: Biden secretly linked to Chinese financial influence operation

 ### Created new session: debug_session_id

User > Headline: Biden secretly linked to Chinese financial influence operation
Source: Conservative Brief
Author: Unknown
Date: Unknown

Body:
President Joe Biden has responded to Republicans’ claims that they have evidence that the Biden family received indirect payments from a Chinese company. Advertisement It happened on Friday night when an agitated President Biden responded to Russian President Vladimir Putin having an arrest warrant issued for him by the International Criminal Court as a reporter snuck in a question about his family’s business dealings.

Q Could you give us your reaction to the International Criminal Court issuing an arrest warrant for Vladimir Putin?
THE PRESIDENT: Well, I think it’s justified. But the question is, it’s not recognized internationally by us either. But I think it makes a very st

In [19]:
await process_batch(all_articles[8:10], "Batch 5")

=== Processing Batch 5 (2 articles) ===

[Batch 5] Article 1: How Modern 3D Laser Scanning Services Are Transforming Data Capture in Technology-Driven Industries

 ### Created new session: debug_session_id

User > Headline: How Modern 3D Laser Scanning Services Are Transforming Data Capture in Technology-Driven Industries
Source: News Examiner
Author: Jimmy Rustling
Date: 11/30/2025

Body:
In the fast-moving technology sector, 3D laser scanning services are redefining how companies collect, analyze, and apply digital information from physical objects. Using advanced laser scanning devices, teams can rapidly capture data from complex environments, producing a detailed point cloud that reflects real-world conditions with exceptional precision.

Precision Data for Engineering and Industrial Applications
Reverse Engineering and Advanced Digital Workflows
Reducing Project Costs and Improving Planning Accuracy

Compared to traditional measuring equipment, this cutting edge technology drastic

In [20]:
await process_batch(all_articles[10:12], "Batch 6")

=== Processing Batch 6 (2 articles) ===

[Batch 6] Article 1: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage

 ### Created new session: debug_session_id

User > Headline: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage
Source: Fox News
Author: Madison Colombo
Date: 11/11/2025

Body:
The International Olympic Committee is preparing to ban transgender women from female Olympic events following a scientific review concluding that biological males retain permanent physical advantages.
Former Olympic champion Caitlyn Jenner, a transgender woman, voiced support for restrictions, stating that biological males possess athletic advantages that hormone therapy cannot entirely eliminate.

JENNER'S POSITION
“I am a trans woman, but I am still biologically male,” Jenner said on 'America Reports.' She argued the policy is necessary to protect fairness in women’s sports.
Jenner noted tha

In [21]:
await process_batch(all_articles[12:14], "Batch 7")

=== Processing Batch 7 (2 articles) ===

[Batch 7] Article 1: Death toll in Hong Kong apartment complex blaze rises to 146 as the city mourns

 ### Created new session: debug_session_id

User > Headline: Death toll in Hong Kong apartment complex blaze rises to 146 as the city mourns
Source: AP News
Author: David Rising
Date: 11/30/2025

Body:
Hong Kong’s apartment complex fire death toll rose to 146 as investigators continued searching the burned buildings of the Wang Fuk Court complex. Many victims were found in units and on rooftops.
The Disaster Victim Identification Unit is conducting slow and meticulous searches due to low visibility inside the damaged towers.
Four of the seven blocks have been fully examined.

COMMUNITY RESPONSE
Mourners formed long lines to place flowers at the scene. Some described the tragedy as a wake-up call about fire safety in high-rise buildings.
Many residents lost all belongings. Donations and support have poured in for displaced families.

INVESTIGATIO

In [22]:
await process_batch(all_articles[14:16], "Batch 8")

=== Processing Batch 8 (2 articles) ===

[Batch 8] Article 1: U.S. pauses green card, citizenship applications for people from 19 countries

 ### Created new session: debug_session_id

User > Headline: U.S. pauses green card, citizenship applications for people from 19 countries
Source: NPR
Author: Ximena Bustillo
Date: 12/03/2025

Body:
The Department of Homeland Security announced that U.S. Citizenship and Immigration Services (USCIS) will pause processing all pending immigration applications — including green cards, citizenship, and asylum — from individuals originating from 19 countries included in a recent travel ban.
The change follows a shooting near the White House in which two National Guard members were killed by an Afghan national.

TRAVEL BAN AND RE-REVIEW PROCESS
President Trump enacted the travel ban in June, targeting 12 countries fully and restricting seven more after a firebombing attack in Colorado. Now, USCIS says immigrants from all 19 countries will have their case

In [23]:
await process_batch(all_articles[16:18], "Batch 9")

=== Processing Batch 9 (2 articles) ===

[Batch 9] Article 1: Proposal for 'healing tsar' to reunite Britain after Brexit

 ### Created new session: debug_session_id

User > Headline: Proposal for 'healing tsar' to reunite Britain after Brexit
Source: The Guardian
Author: Unknown
Date: 04/01/2019

Body:
A working group composed of representatives from major UK institutions has proposed creating a 'healing tsar' to help reunify Britain following the deep social divisions caused by Brexit.

THE ROLE OF A HEALING TSAR
The idea, inspired by discussions reportedly initiated by Prince Charles, would place a unifying figure at the center of national reconciliation efforts. Several well-known public figures have been informally considered.
Music, community events, and symbolic gestures are being evaluated as part of a broad cultural healing strategy.

BEHIND-THE-SCENES TENSIONS
Meetings at Thenford House — the estate of Michael Heseltine — have revealed competing interests among political part

In [24]:
await process_batch(all_articles[18:], "Batch 10")

=== Processing Batch 10 (2 articles) ===

[Batch 10] Article 1: Winds, warmth and relentless drought fueled Los Angeles fires, scientists say

 ### Created new session: debug_session_id

User > Headline: Winds, warmth and relentless drought fueled Los Angeles fires, scientists say
Source: Reuters
Author: Alison Withers and David Stanway
Date: 01/08/2025

Body:
Scientists say that the wind-driven wildfires sweeping across Los Angeles reflect escalating climate-related weather extremes.
Erupting outside the traditional fire season, the blazes spread rapidly as extreme winds, warm temperatures, and long-term drought combined to create dangerous conditions.

CLIMATE CONTEXT
California’s fire risk is now considered year-round. As temperatures rise and vegetation dries, massive fires have become more frequent.
Kimberley Simpson of the University of Sheffield explained that climate change is reshaping regional wildfire patterns.

WIND DYNAMICS
The winds fueling these fires differ from typical

In [ ]:
async def main():
    runner = InMemoryRunner(agent=root_agent)
    prompt = "Hello, how does this work?"
    response = await runner.run_debug(prompt)
    
    print(response)

if __name__ == "__main__":
    asyncio.run(main())